# LLM Free-Features Joint ML+DL Ensembles

Ноутбук собирает общий ансамбль free-features ML и DL моделей.

Источники:
- ML: результаты provider ensembles для free-features моделей.
- DL: результаты provider ensembles для free-features DL моделей.

Новые базовые модели здесь не обучаются: используются сохраненные predictions, затем подбирается общий ML+DL ансамбль.


In [ ]:

import warnings
warnings.filterwarnings("ignore")

import os
import json
import time
import math
from datetime import datetime
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.optimize import nnls

from sklearn.linear_model import (
    LinearRegression,
    Lasso,
    ElasticNet,
    Ridge,
    BayesianRidge,
    HuberRegressor,
    QuantileRegressor,
)
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor,
)

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    XGBRegressor = None
    print(f"xgboost недоступен, xgb stacker пропущен: {e}")


In [ ]:
# ------------------------------------------------------------
# Global config
# ------------------------------------------------------------
target_col = "duration_hours"
cap = 960
seed = 2
np.random.seed(seed)

DATA_PATH_ENV = "DATA_FINALL_PATH"
ARTIFACT_ROOT_ENV = "JOINT_FREE_FEATURES_ARTIFACTS_DIR"


def require_path(env_name: str, label: str) -> Path:
    raw_value = os.environ.get(env_name, "").strip()
    if not raw_value:
        raise FileNotFoundError(f"Укажи путь к {label} через переменную окружения {env_name}.")
    path = Path(raw_value).expanduser()
    if not path.exists():
        raise FileNotFoundError(f"Файл для {label} не найден: {path}")
    return path


DATA_PATH = require_path(DATA_PATH_ENV, "data_finall")
ARTIFACT_ROOT = Path(
    os.environ.get(ARTIFACT_ROOT_ENV, "./artifacts_free_features_joint_ml_dl_ensembles")
).expanduser()
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_NAME = "run_" + datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = ARTIFACT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

ML_RUN_DIR_MANUAL = None
DL_RUN_DIR_MANUAL = None
ML_ARTIFACT_ROOT_MANUAL = None
DL_ARTIFACT_ROOT_MANUAL = None

ML_RUN_DIR_ENV = "ML_PROVIDER_FUSION_RUN_DIR"
DL_RUN_DIR_ENV = "DL_PROVIDER_FUSION_RUN_DIR"
ML_ROOT_ENV = "ML_PROVIDER_FUSION_ARTIFACT_ROOT"
DL_ROOT_ENV = "DL_PROVIDER_FUSION_ARTIFACT_ROOT"

ML_ARTIFACT_ROOT_CANDIDATES = []
DL_ARTIFACT_ROOT_CANDIDATES = []

STANDARD_RUN_FILES = {
    "catalog": "catalog.csv",
    "blend_predictions": "blend_predictions.csv",
    "holdout_predictions": "holdout_predictions.csv",
    "summary": "provider_ensemble_summary.csv",
}

ML_FREE_FEATURES_RUN_FILES = {
    **STANDARD_RUN_FILES,
    "catalog": "free_features_catalog.csv",
    "blend_predictions": "free_features_blend_predictions.csv",
    "holdout_predictions": "free_features_holdout_predictions.csv",
    "summary": "free_features_provider_ensemble_summary.csv",
}

REQUIRED_RUN_FILES = STANDARD_RUN_FILES
REQUIRED_RUN_FILES_BY_SOURCE = {
    "ML": ML_FREE_FEATURES_RUN_FILES,
    "DL": STANDARD_RUN_FILES,
}

BLEND_TRAIN_FRAC_EXPECTED = 0.80
BLEND_FIT_FRAC = 0.50
STRICT_INDEX_ALIGNMENT = True

SOURCE_TOP_N = 6
JOINT_EXHAUSTIVE_TOP_N = 24
MAX_COMBO_SIZE = 4
INCLUDE_SOURCE_LEVEL_CANDIDATES = True
INCLUDE_PROVIDER_LEVEL_CANDIDATES = True

DO_FINAL_REFIT_TOP_N = 20
DO_EXHAUSTIVE_SEARCH = True
DO_STACKERS = True


In [ ]:

# ------------------------------------------------------------
# Save helpers
# ------------------------------------------------------------
def save_df(df: pd.DataFrame, name: str):
    run_path = RUN_DIR / name
    latest_name = name.replace(".csv", "_latest.csv") if name.endswith(".csv") else name
    latest_path = ARTIFACT_ROOT / latest_name
    df.to_csv(run_path, index=False)
    df.to_csv(latest_path, index=False)
    return run_path, latest_path

def save_json(payload, name: str):
    run_path = RUN_DIR / name
    latest_name = name.replace(".json", "_latest.json") if name.endswith(".json") else name
    latest_path = ARTIFACT_ROOT / latest_name
    run_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    latest_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return run_path, latest_path

def clip_pred(pred):
    return np.clip(np.asarray(pred, dtype=float), 0, None)


In [ ]:

# ------------------------------------------------------------
# Resolve ML and DL source run dirs
# ------------------------------------------------------------
def _to_path_or_none(value):
    if value is None:
        return None
    value = str(value).strip()
    if not value:
        return None
    return Path(value).expanduser()

def _dedupe_paths(paths):
    seen = set()
    out = []
    for p in paths:
        if p is None:
            continue
        p = Path(p)
        key = str(p.resolve()) if p.exists() else str(p)
        if key not in seen:
            out.append(p)
            seen.add(key)
    return out

def read_json_if_exists(path: Path):
    if path is None or not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def is_valid_run_dir(path: Path) -> bool:
    path = Path(path)
    return path.exists() and path.is_dir() and (all((path / fname).exists() for fname in STANDARD_RUN_FILES) or all((path / fname).exists() for fname in ML_FREE_FEATURES_RUN_FILES))

def latest_valid_run_under_root(root: Path):
    root = Path(root)
    if not root.exists() or not root.is_dir():
        return None, []
    valid = []
    if is_valid_run_dir(root):
        valid.append(root)
    for child in root.iterdir():
        if child.is_dir() and child.name.startswith("run_") and is_valid_run_dir(child):
            valid.append(child)
    if not valid:
        return None, []
    valid = sorted(valid, key=lambda p: p.stat().st_mtime, reverse=True)
    return valid[0], valid

def resolve_run_dir(
    label: str,
    manual_run_dir,
    run_dir_env_var: str,
    manual_root,
    root_env_var: str,
    root_candidates,
):
    resolution_rows = []

    def _log(path, source, kind, valid):
        resolution_rows.append({
            "label": label,
            "source": source,
            "kind": kind,
            "path": str(path) if path is not None else None,
            "exists": bool(path is not None and Path(path).exists()),
            "valid_run_dir": bool(valid),
        })

    # 1) exact run dir: manual
    p = _to_path_or_none(manual_run_dir)
    if p is not None:
        valid = is_valid_run_dir(p)
        _log(p, "manual_run_dir", "run_dir", valid)
        if valid:
            return p, pd.DataFrame(resolution_rows)

    # 2) exact run dir: env
    p = _to_path_or_none(os.environ.get(run_dir_env_var))
    if p is not None:
        valid = is_valid_run_dir(p)
        _log(p, f"env:{run_dir_env_var}", "run_dir", valid)
        if valid:
            return p, pd.DataFrame(resolution_rows)

    # 3) roots: manual root -> env root -> candidate roots
    roots = []
    roots.append(_to_path_or_none(manual_root))
    roots.append(_to_path_or_none(os.environ.get(root_env_var)))
    roots.extend([_to_path_or_none(x) for x in root_candidates])
    roots = _dedupe_paths(roots)

    for idx, root in enumerate(roots, start=1):
        if root is None:
            continue
        best_run, all_valid = latest_valid_run_under_root(root)
        _log(root, f"root_probe_{idx}", "artifact_root", best_run is not None)
        if best_run is not None:
            _log(best_run, f"root_probe_{idx}:best_run", "run_dir", True)
            return best_run, pd.DataFrame(resolution_rows)

    return None, pd.DataFrame(resolution_rows)

ML_RUN_DIR, ml_resolution_df = resolve_run_dir(
    label="ML",
    manual_run_dir=ML_RUN_DIR_MANUAL,
    run_dir_env_var=ML_RUN_DIR_ENV,
    manual_root=ML_ARTIFACT_ROOT_MANUAL,
    root_env_var=ML_ROOT_ENV,
    root_candidates=ML_ARTIFACT_ROOT_CANDIDATES,
)

DL_RUN_DIR, dl_resolution_df = resolve_run_dir(
    label="DL",
    manual_run_dir=DL_RUN_DIR_MANUAL,
    run_dir_env_var=DL_RUN_DIR_ENV,
    manual_root=DL_ARTIFACT_ROOT_MANUAL,
    root_env_var=DL_ROOT_ENV,
    root_candidates=DL_ARTIFACT_ROOT_CANDIDATES,
)

resolution_df = pd.concat([ml_resolution_df, dl_resolution_df], ignore_index=True)
save_df(resolution_df, "source_run_resolution_log.csv")

if ML_RUN_DIR is None:
    raise FileNotFoundError(
        "Не удалось найти ML provider-fusion run dir. "
        "Заполни ML_RUN_DIR_MANUAL или ML_ARTIFACT_ROOT_MANUAL."
    )

if DL_RUN_DIR is None:
    raise FileNotFoundError(
        "Не удалось найти DL provider-fusion run dir. "
        "Заполни DL_RUN_DIR_MANUAL или DL_ARTIFACT_ROOT_MANUAL."
    )

print("ML run dir:", ML_RUN_DIR)
print("DL run dir:", DL_RUN_DIR)
display(resolution_df)


In [ ]:

# ------------------------------------------------------------
# Load source run artifacts and harmonize metadata
# ------------------------------------------------------------
def _experiment_to_scheme_family(experiment_name: str):
    if experiment_name == "llm_only":
        return "llm_only"
    if str(experiment_name).startswith("cluster_before_llm__"):
        return "cluster_before_llm"
    if str(experiment_name).startswith("llm_then_cluster__"):
        return "llm_then_cluster"
    return None

def _ml_cache_name(base_id: str):
    return str(base_id).replace("::", "__") + ".npy"

def _read_ml_free_features_prediction_frame(run_dir: Path, base_ids, cache_split: str):
    cache_dir = run_dir / "pred_cache" / cache_split
    cols = {}
    missing = []
    for base_id in base_ids:
        path = cache_dir / _ml_cache_name(base_id)
        if not path.exists():
            missing.append(str(path))
            continue
        cols[base_id] = np.load(path)
    if missing:
        raise FileNotFoundError(
            f"ML free-features pred_cache is incomplete for split={cache_split}. "
            f"First missing files: {missing[:5]}"
        )
    return pd.DataFrame(cols)

def _load_ml_free_features_bundle(source_group: str, run_dir: Path, manifest, summary):
    base_meta = pd.read_csv(run_dir / "refit_top6_base_metrics.csv")
    if "base_id" not in base_meta.columns:
        raise KeyError(f"{source_group}: refit_top6_base_metrics.csv does not contain base_id")

    base_meta["base_id"] = base_meta["base_id"].astype(str)
    base_ids = base_meta["base_id"].tolist()

    pred_blend_val = _read_ml_free_features_prediction_frame(run_dir, base_ids, "blend_val")
    pred_test_typ = _read_ml_free_features_prediction_frame(run_dir, base_ids, "fullfit_test_typical")
    pred_test_full = _read_ml_free_features_prediction_frame(run_dir, base_ids, "fullfit_test_full")

    if "cv_MAE_internal" not in base_meta.columns and "source_cv_MAE_internal" in base_meta.columns:
        base_meta["cv_MAE_internal"] = base_meta["source_cv_MAE_internal"]
    if "holdout_typical_MAE" not in base_meta.columns and "source_holdout_typical_MAE" in base_meta.columns:
        base_meta["holdout_typical_MAE"] = base_meta["source_holdout_typical_MAE"]
    if "holdout_full_MAE" not in base_meta.columns and "source_holdout_full_MAE" in base_meta.columns:
        base_meta["holdout_full_MAE"] = base_meta["source_holdout_full_MAE"]
    if "scheme_family" not in base_meta.columns:
        base_meta["scheme_family"] = base_meta["experiment"].map(_experiment_to_scheme_family)
    if "epochs_hint" not in base_meta.columns:
        base_meta["epochs_hint"] = np.nan

    return _finalize_source_bundle(
        source_group=source_group,
        run_dir=run_dir,
        manifest=manifest,
        summary=summary,
        base_meta=base_meta,
        pred_blend_val=pred_blend_val,
        pred_test_typ=pred_test_typ,
        pred_test_full=pred_test_full,
    )

def _finalize_source_bundle(source_group: str, run_dir: Path, manifest, summary, base_meta, pred_blend_val, pred_test_typ, pred_test_full):
    if "base_id" not in base_meta.columns:
        raise KeyError(f"{source_group}: metadata does not contain base_id")

    base_meta["base_id"] = base_meta["base_id"].astype(str)

    available = [
        b for b in base_meta["base_id"].tolist()
        if b in pred_blend_val.columns
        and b in pred_test_typ.columns
        and b in pred_test_full.columns
    ]
    if len(available) == 0:
        raise RuntimeError(f"{source_group}: no common base_ids between meta and prediction matrices")

    rename_map = {b: f"{source_group}::{b}" for b in available}

    base_meta = base_meta[base_meta["base_id"].isin(available)].copy()
    base_meta.insert(1, "source_group", source_group)
    base_meta.insert(2, "source_run_dir", str(run_dir))
    base_meta.insert(3, "base_id_original", base_meta["base_id"].astype(str))
    base_meta["base_id"] = base_meta["base_id"].map(rename_map)

    if "provider" not in base_meta.columns:
        base_meta["provider"] = None
    if "scheme_family" not in base_meta.columns:
        base_meta["scheme_family"] = None
    if "experiment" not in base_meta.columns:
        base_meta["experiment"] = None
    if "model" not in base_meta.columns:
        base_meta["model"] = None
    if "cv_MAE_internal" not in base_meta.columns:
        base_meta["cv_MAE_internal"] = np.nan
    if "holdout_typical_MAE" not in base_meta.columns:
        base_meta["holdout_typical_MAE"] = np.nan
    if "holdout_full_MAE" not in base_meta.columns:
        base_meta["holdout_full_MAE"] = np.nan
    if "epochs_hint" not in base_meta.columns:
        base_meta["epochs_hint"] = np.nan

    pred_blend_val = pred_blend_val[available].rename(columns=rename_map)
    pred_test_typ = pred_test_typ[available].rename(columns=rename_map)
    pred_test_full = pred_test_full[available].rename(columns=rename_map)

    for name, df_ in [
        ("blend_val", pred_blend_val),
        ("test_typical", pred_test_typ),
        ("test_full", pred_test_full),
    ]:
        if df_.isna().any().any():
            raise RuntimeError(f"{source_group}: found NaNs in {name} prediction matrix")

    return {
        "source_group": source_group,
        "run_dir": run_dir,
        "manifest": manifest,
        "summary": summary,
        "base_meta": base_meta.reset_index(drop=True),
        "pred_blend_val": pred_blend_val.reset_index(drop=True),
        "pred_test_typical": pred_test_typ.reset_index(drop=True),
        "pred_test_full": pred_test_full.reset_index(drop=True),
    }

def load_source_bundle(source_group: str, run_dir: Path):
    run_dir = Path(run_dir)

    manifest = read_json_if_exists(run_dir / "run_manifest.json")
    summary = read_json_if_exists(run_dir / "best_ensemble_summary.json")

    if all((run_dir / fname).exists() for fname in STANDARD_RUN_FILES):
        base_meta = pd.read_csv(run_dir / "available_base_meta.csv")
        pred_blend_val = pd.read_csv(run_dir / "base_pred_blend_val.csv")
        pred_test_typ = pd.read_csv(run_dir / "base_pred_fullfit_test_typical.csv")
        pred_test_full = pd.read_csv(run_dir / "base_pred_fullfit_test_full.csv")
        return _finalize_source_bundle(
            source_group=source_group,
            run_dir=run_dir,
            manifest=manifest,
            summary=summary,
            base_meta=base_meta,
            pred_blend_val=pred_blend_val,
            pred_test_typ=pred_test_typ,
            pred_test_full=pred_test_full,
        )

    if source_group == "ML" and all((run_dir / fname).exists() for fname in ML_FREE_FEATURES_RUN_FILES):
        return _load_ml_free_features_bundle(source_group, run_dir, manifest, summary)

    raise FileNotFoundError(
        f"{source_group}: run dir {run_dir} is neither a standard prediction run "
        f"nor an ML free-features top-6 run."
    )

ml_bundle = load_source_bundle("ML", ML_RUN_DIR)
dl_bundle = load_source_bundle("DL", DL_RUN_DIR)

source_registry_rows = []
for bundle in [ml_bundle, dl_bundle]:
    manifest = bundle.get("manifest") or {}
    summary = bundle.get("summary") or {}
    source_registry_rows.append({
        "source_group": bundle["source_group"],
        "run_dir": str(bundle["run_dir"]),
        "n_base_learners": int(len(bundle["base_meta"])),
        "n_blend_val_rows": int(bundle["pred_blend_val"].shape[0]),
        "n_test_typical_rows": int(bundle["pred_test_typical"].shape[0]),
        "n_test_full_rows": int(bundle["pred_test_full"].shape[0]),
        "blend_train_frac_manifest": manifest.get("blend_train_frac", np.nan),
        "blend_fit_frac_manifest": manifest.get("blend_fit_frac", np.nan),
        "best_single_from_summary": (summary.get("best_single_base_by_val") or summary.get("best_single_model_by_val")),
        "best_ensemble_tag_from_summary": (summary.get("best_ensemble") or {}).get("tag"),
    })

source_registry_df = pd.DataFrame(source_registry_rows)
save_df(source_registry_df, "source_run_registry.csv")
display(source_registry_df)

blend_train_fracs = []
for bundle in [ml_bundle, dl_bundle]:
    manifest = bundle.get("manifest") or {}
    val = manifest.get("blend_train_frac", None)
    if val is not None and not pd.isna(val):
        blend_train_fracs.append(float(val))

if len(set(blend_train_fracs)) > 1:
    raise ValueError(
        f"ML and DL source runs use different blend_train_frac: {blend_train_fracs}"
    )

BLEND_TRAIN_FRAC_SOURCE = blend_train_fracs[0] if blend_train_fracs else BLEND_TRAIN_FRAC_EXPECTED
print("BLEND_TRAIN_FRAC_SOURCE =", BLEND_TRAIN_FRAC_SOURCE)


In [ ]:

# ------------------------------------------------------------
# Reproduce the canonical split and validate row alignment
# ------------------------------------------------------------
df = pd.read_csv(DATA_PATH)

split = int(len(df) * 0.8)
train_full = df.iloc[:split].copy()
test_full = df.iloc[split:].copy()
test_typical = test_full[test_full[target_col] < cap].copy()
train_filt = train_full[train_full[target_col] < cap].copy()

meta_Xtr0 = train_filt.drop(columns=[target_col], errors="ignore").reset_index(drop=True)
meta_ytr0 = train_filt[target_col].reset_index(drop=True)

meta_Xte0 = test_full.drop(columns=[target_col], errors="ignore").reset_index(drop=True)
meta_yte0 = test_full[target_col].reset_index(drop=True)

meta_Xte_typ0 = test_typical.drop(columns=[target_col], errors="ignore").reset_index(drop=True)
meta_yte_typ0 = test_typical[target_col].reset_index(drop=True)

blend_split_idx = int(len(meta_Xtr0) * BLEND_TRAIN_FRAC_SOURCE)
if blend_split_idx <= 0 or blend_split_idx >= len(meta_Xtr0):
    raise ValueError("Некорректный BLEND_TRAIN_FRAC_SOURCE.")

y_blend_val = meta_ytr0.iloc[blend_split_idx:].reset_index(drop=True).to_numpy(dtype=float)
y_test_typ = meta_yte_typ0.to_numpy(dtype=float)
y_test_full = meta_yte0.to_numpy(dtype=float)

shape_check_rows = []
for bundle in [ml_bundle, dl_bundle]:
    shape_check_rows.append({
        "source_group": bundle["source_group"],
        "blend_val_rows_source": int(bundle["pred_blend_val"].shape[0]),
        "blend_val_rows_expected": int(len(y_blend_val)),
        "test_typical_rows_source": int(bundle["pred_test_typical"].shape[0]),
        "test_typical_rows_expected": int(len(y_test_typ)),
        "test_full_rows_source": int(bundle["pred_test_full"].shape[0]),
        "test_full_rows_expected": int(len(y_test_full)),
    })

shape_check_df = pd.DataFrame(shape_check_rows)
save_df(shape_check_df, "source_shape_validation.csv")
display(shape_check_df)

for bundle in [ml_bundle, dl_bundle]:
    if bundle["pred_blend_val"].shape[0] != len(y_blend_val):
        raise ValueError(
            f"{bundle['source_group']}: blend-val row mismatch -> "
            f"{bundle['pred_blend_val'].shape[0]} vs expected {len(y_blend_val)}"
        )
    if bundle["pred_test_typical"].shape[0] != len(y_test_typ):
        raise ValueError(
            f"{bundle['source_group']}: test-typical row mismatch -> "
            f"{bundle['pred_test_typical'].shape[0]} vs expected {len(y_test_typ)}"
        )
    if bundle["pred_test_full"].shape[0] != len(y_test_full):
        raise ValueError(
            f"{bundle['source_group']}: test-full row mismatch -> "
            f"{bundle['pred_test_full'].shape[0]} vs expected {len(y_test_full)}"
        )

print("train_full   :", train_full.shape)
print("train_filt   :", train_filt.shape)
print("test_full    :", test_full.shape)
print("test_typical :", test_typical.shape)
print("blend_split_idx:", blend_split_idx)
print("len(y_blend_val):", len(y_blend_val))
print("len(y_test_typ) :", len(y_test_typ))
print("len(y_test_full):", len(y_test_full))


In [ ]:

# ------------------------------------------------------------
# Combine ML and DL base pools
# ------------------------------------------------------------
base_pool_df = pd.concat(
    [ml_bundle["base_meta"], dl_bundle["base_meta"]],
    ignore_index=True,
    sort=False,
)

val_pred_df = pd.concat(
    [ml_bundle["pred_blend_val"], dl_bundle["pred_blend_val"]],
    axis=1,
)
test_typ_fullfit_pred_df = pd.concat(
    [ml_bundle["pred_test_typical"], dl_bundle["pred_test_typical"]],
    axis=1,
)
test_full_fullfit_pred_df = pd.concat(
    [ml_bundle["pred_test_full"], dl_bundle["pred_test_full"]],
    axis=1,
)

available_base_ids = val_pred_df.columns.tolist()
if len(available_base_ids) < 2:
    raise RuntimeError("Для joint-ensembling нужно хотя бы 2 base learners.")

base_pool_df = (
    base_pool_df
    .set_index("base_id")
    .loc[available_base_ids]
    .reset_index()
)

model_group_map = dict(zip(base_pool_df["base_id"], base_pool_df["source_group"]))
base_id_to_meta = base_pool_df.set_index("base_id").to_dict(orient="index")

save_df(base_pool_df, "combined_available_base_meta.csv")
save_df(val_pred_df, "joint_base_pred_blend_val.csv")
save_df(test_typ_fullfit_pred_df, "joint_base_pred_fullfit_test_typical.csv")
save_df(test_full_fullfit_pred_df, "joint_base_pred_fullfit_test_full.csv")

source_counts_df = (
    base_pool_df.groupby("source_group", dropna=False)
    .agg(
        n_base_learners=("base_id", "count"),
        n_unique_models=("model", "nunique"),
        best_cv_MAE=("cv_MAE_internal", "min"),
        best_holdout_typical_MAE=("holdout_typical_MAE", "min"),
    )
    .reset_index()
)
save_df(source_counts_df, "source_base_counts.csv")

display(
    base_pool_df[
        [
            "base_id",
            "source_group",
            "model",
            "provider",
            "scheme_family",
            "experiment",
            "cv_MAE_internal",
            "holdout_typical_MAE",
            "holdout_full_MAE",
        ]
    ]
    .sort_values(["source_group", "cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"], kind="stable")
    .reset_index(drop=True)
)

display(source_counts_df)

print("Total available base learners:", len(available_base_ids))
print("ML base learners:", int((base_pool_df["source_group"] == "ML").sum()))
print("DL base learners:", int((base_pool_df["source_group"] == "DL").sum()))


In [ ]:

# ------------------------------------------------------------
# Metrics + ensemble helpers
# ------------------------------------------------------------
def mae_metric(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse_metric(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mdae_metric(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.median(np.abs(y_true - y_pred)))

def score_predictions(y_true, y_pred):
    return {
        "MAE": mae_metric(y_true, y_pred),
        "RMSE": rmse_metric(y_true, y_pred),
        "MdAE": mdae_metric(y_true, y_pred),
    }

def modality_counts(models):
    n_ml = int(sum(model_group_map.get(m) == "ML" for m in models))
    n_dl = int(sum(model_group_map.get(m) == "DL" for m in models))
    return n_ml, n_dl, bool(n_ml > 0 and n_dl > 0)

def get_base_leaderboard_from_predictions(pred_df: pd.DataFrame, y_true):
    rows = []
    for col in pred_df.columns:
        meta = base_id_to_meta.get(col, {})
        rows.append({
            "base_id": col,
            "source_group": meta.get("source_group"),
            "model": meta.get("model"),
            "provider": meta.get("provider"),
            "scheme_family": meta.get("scheme_family"),
            "experiment": meta.get("experiment"),
            "MAE": mae_metric(y_true, pred_df[col].values),
            "RMSE": rmse_metric(y_true, pred_df[col].values),
            "MdAE": mdae_metric(y_true, pred_df[col].values),
        })
    return (
        pd.DataFrame(rows)
        .sort_values(["MAE", "RMSE", "MdAE", "base_id"], kind="stable")
        .reset_index(drop=True)
    )

def aggregate_predictions(pred_matrix: np.ndarray, method: str, trim_frac: float = 0.1):
    pred_matrix = np.asarray(pred_matrix, dtype=float)
    if pred_matrix.ndim == 1:
        return clip_pred(pred_matrix)
    if method == "mean":
        out = pred_matrix.mean(axis=1)
    elif method == "median":
        out = np.median(pred_matrix, axis=1)
    elif method == "trimmed_mean":
        m = pred_matrix.shape[1]
        k = int(np.floor(m * trim_frac))
        if k <= 0 or 2 * k >= m:
            out = pred_matrix.mean(axis=1)
        else:
            sorted_mat = np.sort(pred_matrix, axis=1)
            out = sorted_mat[:, k:m-k].mean(axis=1)
    else:
        raise KeyError(method)
    return clip_pred(out)

def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    w = np.where(np.isfinite(w), w, 0.0)
    w = np.clip(w, 0, None)
    s = w.sum()
    if s <= 0:
        return np.ones_like(w) / len(w)
    return w / s

def weights_from_errors(pred_fit: pd.DataFrame, y_fit, scheme: str, temperature: float = 1.0, power: float = 1.0):
    errs = np.array([mae_metric(y_fit, pred_fit[c].values) for c in pred_fit.columns], dtype=float)
    if scheme == "equal":
        w = np.ones(len(errs), dtype=float)
    elif scheme == "inverse_mae":
        w = 1.0 / np.maximum(errs, 1e-9)
    elif scheme == "inverse_mae_sq":
        w = 1.0 / np.maximum(errs, 1e-9) ** 2
    elif scheme == "softmax_mae":
        z = -errs / max(float(temperature), 1e-6)
        z = z - z.max()
        w = np.exp(z)
    elif scheme == "rank_power":
        order = np.argsort(errs)
        ranks = np.empty_like(order)
        ranks[order] = np.arange(1, len(errs) + 1)
        w = 1.0 / np.power(ranks, power)
    else:
        raise KeyError(scheme)
    return normalize_weights(w)

def fit_weighted_blender(pred_fit: pd.DataFrame, y_fit, scheme: str, **kwargs):
    X = pred_fit.values.astype(float)
    if scheme in {"equal", "inverse_mae", "inverse_mae_sq", "softmax_mae", "rank_power"}:
        w = weights_from_errors(pred_fit, y_fit, scheme, **kwargs)
        return {"kind": "weights", "weights": w}
    elif scheme == "nnls":
        w, _ = nnls(X, np.asarray(y_fit, dtype=float))
        w = normalize_weights(w)
        return {"kind": "weights", "weights": w}
    elif scheme == "simplex_mae":
        n = X.shape[1]
        x0 = np.ones(n, dtype=float) / n
        bounds = [(0.0, 1.0)] * n
        cons = {"type": "eq", "fun": lambda w: np.sum(w) - 1.0}

        def objective(w):
            return mae_metric(y_fit, X @ w)

        res = minimize(
            objective,
            x0=x0,
            method="SLSQP",
            bounds=bounds,
            constraints=[cons],
            options={"maxiter": 500, "ftol": 1e-8},
        )
        w = normalize_weights(res.x if res.success else x0)
        return {"kind": "weights", "weights": w, "optimizer_success": bool(res.success)}
    else:
        raise KeyError(scheme)

def predict_weighted_blender(fitted, pred_df: pd.DataFrame):
    w = np.asarray(fitted["weights"], dtype=float)
    return clip_pred(pred_df.values @ w)

def build_stacker(stacker_name: str):
    if stacker_name == "linear":
        return LinearRegression()
    elif stacker_name == "positive_linear":
        return LinearRegression(positive=True)
    elif stacker_name == "ridge":
        return Ridge(alpha=1.0)
    elif stacker_name == "lasso":
        return Lasso(alpha=1e-3, max_iter=20000, random_state=seed)
    elif stacker_name == "elastic":
        return ElasticNet(alpha=1e-3, l1_ratio=0.5, max_iter=20000, random_state=seed)
    elif stacker_name == "bayes":
        return BayesianRidge()
    elif stacker_name == "huber":
        return HuberRegressor(max_iter=2000, epsilon=1.35)
    elif stacker_name == "quantile":
        return QuantileRegressor(quantile=0.5, alpha=1e-4, solver="highs")
    elif stacker_name == "gbr":
        return GradientBoostingRegressor(
            loss="absolute_error",
            random_state=seed,
            n_estimators=300,
            learning_rate=0.05,
            max_depth=2,
            min_samples_leaf=5,
            min_samples_split=4,
            subsample=0.9,
        )
    elif stacker_name == "rf":
        return RandomForestRegressor(
            n_estimators=500,
            random_state=seed,
            min_samples_leaf=2,
            n_jobs=-1,
        )
    elif stacker_name == "etr":
        return ExtraTreesRegressor(
            n_estimators=500,
            random_state=seed,
            min_samples_leaf=2,
            n_jobs=-1,
        )
    elif stacker_name == "xgb":
        if not HAS_XGB:
            raise RuntimeError("xgboost is not available")
        return XGBRegressor(
            objective="reg:absoluteerror",
            eval_metric="mae",
            n_estimators=500,
            learning_rate=0.03,
            max_depth=2,
            min_child_weight=3,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_alpha=0.0,
            reg_lambda=1.0,
            tree_method="hist",
            n_jobs=-1,
            random_state=seed,
            verbosity=0,
        )
    else:
        raise KeyError(stacker_name)

def fit_stacker(pred_fit: pd.DataFrame, y_fit, stacker_name: str):
    model = build_stacker(stacker_name)
    model.fit(pred_fit.values.astype(float), np.asarray(y_fit, dtype=float))
    return {"kind": "stacker", "model": model, "stacker_name": stacker_name}

def predict_stacker(fitted, pred_df: pd.DataFrame):
    return clip_pred(fitted["model"].predict(pred_df.values.astype(float)))

def fit_pair_weight_grid(pred_fit: pd.DataFrame, y_fit):
    if pred_fit.shape[1] != 2:
        raise ValueError("pair_grid expects exactly 2 models")
    best = None
    a = pred_fit.iloc[:, 0].values.astype(float)
    b = pred_fit.iloc[:, 1].values.astype(float)
    for w in PAIR_WEIGHT_GRID:
        pred = clip_pred(w * a + (1.0 - w) * b)
        score = mae_metric(y_fit, pred)
        if best is None or score < best["selection_MAE"]:
            best = {"kind": "pair_grid", "weight_first": float(w), "selection_MAE": score}
    return best

def predict_pair_weight_grid(fitted, pred_df: pd.DataFrame):
    w = float(fitted["weight_first"])
    a = pred_df.iloc[:, 0].values.astype(float)
    b = pred_df.iloc[:, 1].values.astype(float)
    return clip_pred(w * a + (1.0 - w) * b)

def top_models_by_fit_mae(pred_fit_df: pd.DataFrame, y_fit):
    fit_rank_df = get_base_leaderboard_from_predictions(pred_fit_df, y_fit)
    ranked_models = fit_rank_df["base_id"].tolist()
    return ranked_models, fit_rank_df

def spec_tag(spec):
    models_part = "__".join(spec["models"])
    return f'{spec["family"]}::{spec["name"]}::{models_part}'

def evaluate_spec_on_selection(spec, pred_fit_df, y_fit, pred_sel_df, y_sel):
    models = spec["models"]
    fit_sub = pred_fit_df[models].copy()
    sel_sub = pred_sel_df[models].copy()

    family = spec["family"]
    name = spec["name"]

    if family == "single":
        pred_sel = clip_pred(sel_sub.iloc[:, 0].values)
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "aggregate":
        pred_sel = aggregate_predictions(sel_sub.values, spec["agg_method"], spec.get("trim_frac", 0.1))
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "weighted":
        fitted = fit_weighted_blender(fit_sub, y_fit, spec["weight_scheme"], **spec.get("weight_kwargs", {}))
        pred_sel = predict_weighted_blender(fitted, sel_sub)
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "weights": fitted.get("weights", []).tolist() if isinstance(fitted.get("weights", []), np.ndarray) else fitted.get("weights", []),
        }
    elif family == "pair_grid":
        fitted = fit_pair_weight_grid(fit_sub, y_fit)
        pred_sel = predict_pair_weight_grid(fitted, sel_sub)
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "weight_first": fitted["weight_first"],
        }
    elif family == "stacker":
        fitted = fit_stacker(fit_sub, y_fit, spec["stacker_name"])
        pred_sel = predict_stacker(fitted, sel_sub)
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "stacker_name": spec["stacker_name"],
        }
    else:
        raise KeyError(family)

    sel_metrics = score_predictions(y_sel, pred_sel)
    n_ml, n_dl, is_mixed = modality_counts(models)
    out["selection_MAE"] = sel_metrics["MAE"]
    out["selection_RMSE"] = sel_metrics["RMSE"]
    out["selection_MdAE"] = sel_metrics["MdAE"]
    out["n_ml_models"] = n_ml
    out["n_dl_models"] = n_dl
    out["is_mixed"] = is_mixed
    out["tag"] = spec_tag(spec)
    out["spec"] = spec
    return out

def refit_and_eval_spec(spec, pred_val_df, y_val, pred_test_typ_df, y_test_typ, pred_test_full_df, y_test_full):
    models = spec["models"]
    val_sub = pred_val_df[models].copy()
    test_typ_sub = pred_test_typ_df[models].copy()
    test_full_sub = pred_test_full_df[models].copy()

    family = spec["family"]
    name = spec["name"]

    if family == "single":
        pred_typ = clip_pred(test_typ_sub.iloc[:, 0].values)
        pred_full = clip_pred(test_full_sub.iloc[:, 0].values)
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "aggregate":
        pred_typ = aggregate_predictions(test_typ_sub.values, spec["agg_method"], spec.get("trim_frac", 0.1))
        pred_full = aggregate_predictions(test_full_sub.values, spec["agg_method"], spec.get("trim_frac", 0.1))
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "weighted":
        fitted = fit_weighted_blender(val_sub, y_val, spec["weight_scheme"], **spec.get("weight_kwargs", {}))
        pred_typ = predict_weighted_blender(fitted, test_typ_sub)
        pred_full = predict_weighted_blender(fitted, test_full_sub)
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "weights": fitted.get("weights", []).tolist() if isinstance(fitted.get("weights", []), np.ndarray) else fitted.get("weights", []),
        }
    elif family == "pair_grid":
        fitted = fit_pair_weight_grid(val_sub, y_val)
        pred_typ = predict_pair_weight_grid(fitted, test_typ_sub)
        pred_full = predict_pair_weight_grid(fitted, test_full_sub)
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "weight_first": fitted["weight_first"],
            "weights_like": [fitted["weight_first"], 1.0 - fitted["weight_first"]],
        }
    elif family == "stacker":
        fitted = fit_stacker(val_sub, y_val, spec["stacker_name"])
        pred_typ = predict_stacker(fitted, test_typ_sub)
        pred_full = predict_stacker(fitted, test_full_sub)
        weights_like = None
        if hasattr(fitted["model"], "coef_"):
            coef = np.ravel(getattr(fitted["model"], "coef_"))
            if len(coef) == len(models):
                weights_like = coef.tolist()
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "weights_like": weights_like,
        }
    else:
        raise KeyError(family)

    typ_metrics = score_predictions(y_test_typ, pred_typ)
    full_metrics = score_predictions(y_test_full, pred_full)
    n_ml, n_dl, is_mixed = modality_counts(models)

    out["tag"] = spec_tag(spec)
    out["n_ml_models"] = n_ml
    out["n_dl_models"] = n_dl
    out["is_mixed"] = is_mixed
    out["MAE_typical"] = typ_metrics["MAE"]
    out["RMSE_typical"] = typ_metrics["RMSE"]
    out["MdAE_typical"] = typ_metrics["MdAE"]
    out["MAE_full"] = full_metrics["MAE"]
    out["RMSE_full"] = full_metrics["RMSE"]
    out["MdAE_full"] = full_metrics["MdAE"]
    return out

def build_greedy_forward_specs(ranked_models, pred_fit_df, y_fit):
    specs = []
    search_modes = [
        ("aggregate", "greedy_mean", {"agg_method": "mean"}),
        ("weighted", "greedy_inverse", {"weight_scheme": "inverse_mae"}),
        ("weighted", "greedy_nnls", {"weight_scheme": "nnls"}),
        ("weighted", "greedy_simplex", {"weight_scheme": "simplex_mae"}),
    ]

    for family, prefix, extra in search_modes:
        current = []
        remaining = ranked_models.copy()

        while remaining:
            best = None
            for m in remaining:
                cand = current + [m]
                fit_sub = pred_fit_df[cand].copy()

                if len(cand) == 1:
                    pred = clip_pred(fit_sub.iloc[:, 0].values)
                elif family == "aggregate":
                    pred = aggregate_predictions(fit_sub.values, extra["agg_method"], extra.get("trim_frac", 0.1))
                elif family == "weighted":
                    fitted = fit_weighted_blender(fit_sub, y_fit, extra["weight_scheme"], **extra.get("weight_kwargs", {}))
                    pred = predict_weighted_blender(fitted, fit_sub)
                else:
                    raise KeyError(family)

                score = mae_metric(y_fit, pred)
                if best is None or score < best["score"]:
                    best = {"models": cand.copy(), "score": score}

            current = best["models"]
            remaining = [m for m in remaining if m not in current]

            if len(current) >= 2:
                spec = {"family": family, "name": f"{prefix}_step{len(current)}", "models": current.copy()}
                spec.update(extra)
                specs.append(spec)

    return specs


In [ ]:
# ------------------------------------------------------------
# Search over strictly cross-modal ML + DL blending / stacking specs
# ------------------------------------------------------------
base_leaderboard = get_base_leaderboard_from_predictions(val_pred_df, y_blend_val)
display(base_leaderboard.head(30))
save_df(base_leaderboard, "joint_base_leaderboard.csv")

blend_cut = int(len(y_blend_val) * BLEND_FIT_FRAC)
if blend_cut <= 0 or blend_cut >= len(y_blend_val):
    raise ValueError("Некорректный BLEND_FIT_FRAC.")

pred_fit_df = val_pred_df.iloc[:blend_cut].copy()
pred_sel_df = val_pred_df.iloc[blend_cut:].copy()
y_fit = y_blend_val[:blend_cut].copy()
y_sel = y_blend_val[blend_cut:].copy()

ranked_models, fit_rank_df = top_models_by_fit_mae(pred_fit_df, y_fit)
save_df(fit_rank_df, "blend_fit_model_ranking.csv")
display(fit_rank_df.head(30))

ranked_ml_models = [m for m in ranked_models if model_group_map.get(m) == "ML"]
ranked_dl_models = [m for m in ranked_models if model_group_map.get(m) == "DL"]

print("Top ML base learners:", ranked_ml_models[:10])
print("Top DL base learners:", ranked_dl_models[:10])
print("Cross-modal-only search: ML+ML / DL+DL / single models are excluded.")

def maybe_add_spec(specs, spec):
    n_ml, n_dl, is_mixed = modality_counts(spec["models"])
    if is_mixed:
        specs.append(spec)

specs = []

# global aggregations on all models: valid only if both modalities exist
all_models = ranked_models.copy()
maybe_add_spec(specs, {"family": "aggregate", "name": "all_mean", "models": all_models, "agg_method": "mean"})
maybe_add_spec(specs, {"family": "aggregate", "name": "all_median", "models": all_models, "agg_method": "median"})
maybe_add_spec(specs, {"family": "aggregate", "name": "all_trim10", "models": all_models, "agg_method": "trimmed_mean", "trim_frac": 0.10})
maybe_add_spec(specs, {"family": "aggregate", "name": "all_trim20", "models": all_models, "agg_method": "trimmed_mean", "trim_frac": 0.20})
maybe_add_spec(specs, {"family": "weighted", "name": "all_inverse_mae", "models": all_models, "weight_scheme": "inverse_mae"})
maybe_add_spec(specs, {"family": "weighted", "name": "all_inverse_mae_sq", "models": all_models, "weight_scheme": "inverse_mae_sq"})
maybe_add_spec(specs, {"family": "weighted", "name": "all_softmax_t1", "models": all_models, "weight_scheme": "softmax_mae", "weight_kwargs": {"temperature": 1.0}})
maybe_add_spec(specs, {"family": "weighted", "name": "all_nnls", "models": all_models, "weight_scheme": "nnls"})
maybe_add_spec(specs, {"family": "weighted", "name": "all_simplex_mae", "models": all_models, "weight_scheme": "simplex_mae"})

# top-k prefixes, but only mixed prefixes are kept
if RUN_TOPK_PREFIX_ENSEMBLES:
    for k in range(2, len(ranked_models) + 1):
        topk = ranked_models[:k]
        if not modality_counts(topk)[2]:
            continue
        maybe_add_spec(specs, {"family": "aggregate", "name": f"top{k}_mean", "models": topk, "agg_method": "mean"})
        maybe_add_spec(specs, {"family": "aggregate", "name": f"top{k}_median", "models": topk, "agg_method": "median"})
        if k >= 5:
            maybe_add_spec(specs, {"family": "aggregate", "name": f"top{k}_trim10", "models": topk, "agg_method": "trimmed_mean", "trim_frac": 0.10})
        maybe_add_spec(specs, {"family": "weighted", "name": f"top{k}_inverse_mae", "models": topk, "weight_scheme": "inverse_mae"})
        maybe_add_spec(specs, {"family": "weighted", "name": f"top{k}_inverse_mae_sq", "models": topk, "weight_scheme": "inverse_mae_sq"})
        for temp in [0.25, 0.5, 1.0, 2.0, 4.0]:
            maybe_add_spec(specs, {"family": "weighted", "name": f"top{k}_softmax_t{temp}", "models": topk, "weight_scheme": "softmax_mae", "weight_kwargs": {"temperature": temp}})
        for power in [0.5, 1.0, 2.0]:
            maybe_add_spec(specs, {"family": "weighted", "name": f"top{k}_rankp{power}", "models": topk, "weight_scheme": "rank_power", "weight_kwargs": {"power": power}})
        maybe_add_spec(specs, {"family": "weighted", "name": f"top{k}_nnls", "models": topk, "weight_scheme": "nnls"})
        maybe_add_spec(specs, {"family": "weighted", "name": f"top{k}_simplex", "models": topk, "weight_scheme": "simplex_mae"})

# all cross-modal pairs only
if RUN_ALL_PAIRS:
    for a, b in combinations(ranked_models, 2):
        if not modality_counts([a, b])[2]:
            continue
        maybe_add_spec(specs, {"family": "aggregate", "name": "pair_mean", "models": [a, b], "agg_method": "mean"})
        maybe_add_spec(specs, {"family": "pair_grid", "name": "pair_grid", "models": [a, b]})
        maybe_add_spec(specs, {"family": "weighted", "name": "pair_nnls", "models": [a, b], "weight_scheme": "nnls"})
        maybe_add_spec(specs, {"family": "weighted", "name": "pair_simplex", "models": [a, b], "weight_scheme": "simplex_mae"})

# triples with at least one ML and one DL
if RUN_ALL_TRIPLES:
    for trio in combinations(ranked_models, 3):
        trio = list(trio)
        if not modality_counts(trio)[2]:
            continue
        maybe_add_spec(specs, {"family": "aggregate", "name": "triple_mean", "models": trio, "agg_method": "mean"})
        maybe_add_spec(specs, {"family": "aggregate", "name": "triple_median", "models": trio, "agg_method": "median"})
        maybe_add_spec(specs, {"family": "weighted", "name": "triple_inverse", "models": trio, "weight_scheme": "inverse_mae"})
        maybe_add_spec(specs, {"family": "weighted", "name": "triple_nnls", "models": trio, "weight_scheme": "nnls"})

# exhaustive top-N subset search, mixed subsets only
if RUN_EXHAUSTIVE_TOPN_SUBSETS:
    topn_models = ranked_models[:min(EXHAUSTIVE_TOP_N, len(ranked_models))]
    for r in range(2, len(topn_models) + 1):
        for subset in combinations(topn_models, r):
            subset = list(subset)
            if not modality_counts(subset)[2]:
                continue
            maybe_add_spec(specs, {"family": "aggregate", "name": f"subset{r}_mean", "models": subset, "agg_method": "mean"})
            maybe_add_spec(specs, {"family": "aggregate", "name": f"subset{r}_median", "models": subset, "agg_method": "median"})
            if r >= 5:
                maybe_add_spec(specs, {"family": "aggregate", "name": f"subset{r}_trim10", "models": subset, "agg_method": "trimmed_mean", "trim_frac": 0.10})
            maybe_add_spec(specs, {"family": "weighted", "name": f"subset{r}_inverse", "models": subset, "weight_scheme": "inverse_mae"})
            maybe_add_spec(specs, {"family": "weighted", "name": f"subset{r}_inverse_sq", "models": subset, "weight_scheme": "inverse_mae_sq"})
            maybe_add_spec(specs, {"family": "weighted", "name": f"subset{r}_softmax", "models": subset, "weight_scheme": "softmax_mae", "weight_kwargs": {"temperature": 1.0}})
            maybe_add_spec(specs, {"family": "weighted", "name": f"subset{r}_nnls", "models": subset, "weight_scheme": "nnls"})
            maybe_add_spec(specs, {"family": "weighted", "name": f"subset{r}_simplex", "models": subset, "weight_scheme": "simplex_mae"})

# greedy forward: keep only mixed specs
if RUN_GREEDY_SEARCH:
    for spec in build_greedy_forward_specs(ranked_models, pred_fit_df, y_fit):
        maybe_add_spec(specs, spec)

# cross-modal prefix grid: top-ML + top-DL
if RUN_CROSS_MODAL_PREFIX_GRID and ranked_ml_models and ranked_dl_models:
    ml_ks = sorted(set([k for k in [1, 2, 3, 4, 5] if k <= len(ranked_ml_models)]))
    dl_ks = sorted(set([k for k in [1, 2, 3, 4, 5, 6, 8, 10] if k <= len(ranked_dl_models)]))

    for km in ml_ks:
        ml_subset = ranked_ml_models[:km]
        for kd in dl_ks:
            dl_subset = ranked_dl_models[:kd]
            subset = ml_subset + dl_subset
            if len(subset) < 2 or not modality_counts(subset)[2]:
                continue
            maybe_add_spec(specs, {"family": "aggregate", "name": f"cross_ml{km}_dl{kd}_mean", "models": subset, "agg_method": "mean"})
            maybe_add_spec(specs, {"family": "aggregate", "name": f"cross_ml{km}_dl{kd}_median", "models": subset, "agg_method": "median"})
            if len(subset) >= 5:
                maybe_add_spec(specs, {"family": "aggregate", "name": f"cross_ml{km}_dl{kd}_trim10", "models": subset, "agg_method": "trimmed_mean", "trim_frac": 0.10})
            maybe_add_spec(specs, {"family": "weighted", "name": f"cross_ml{km}_dl{kd}_inverse", "models": subset, "weight_scheme": "inverse_mae"})
            maybe_add_spec(specs, {"family": "weighted", "name": f"cross_ml{km}_dl{kd}_inverse_sq", "models": subset, "weight_scheme": "inverse_mae_sq"})
            maybe_add_spec(specs, {"family": "weighted", "name": f"cross_ml{km}_dl{kd}_nnls", "models": subset, "weight_scheme": "nnls"})
            maybe_add_spec(specs, {"family": "weighted", "name": f"cross_ml{km}_dl{kd}_simplex", "models": subset, "weight_scheme": "simplex_mae"})
            maybe_add_spec(specs, {"family": "stacker", "name": f"cross_ml{km}_dl{kd}_linear", "models": subset, "stacker_name": "linear"})
            maybe_add_spec(specs, {"family": "stacker", "name": f"cross_ml{km}_dl{kd}_ridge", "models": subset, "stacker_name": "ridge"})
            maybe_add_spec(specs, {"family": "stacker", "name": f"cross_ml{km}_dl{kd}_huber", "models": subset, "stacker_name": "huber"})
            if HAS_XGB:
                maybe_add_spec(specs, {"family": "stacker", "name": f"cross_ml{km}_dl{kd}_xgb", "models": subset, "stacker_name": "xgb"})

# stackers on all / top-k, mixed only
if RUN_STACKERS:
    stackers = ["linear", "positive_linear", "ridge", "lasso", "elastic", "bayes", "huber", "quantile", "gbr", "rf", "etr"]
    if HAS_XGB:
        stackers.append("xgb")
    for stacker_name in stackers:
        maybe_add_spec(specs, {"family": "stacker", "name": f"all_{stacker_name}", "models": all_models, "stacker_name": stacker_name})
        for k in range(2, len(ranked_models) + 1):
            topk = ranked_models[:k]
            if not modality_counts(topk)[2]:
                continue
            maybe_add_spec(specs, {"family": "stacker", "name": f"top{k}_{stacker_name}", "models": topk, "stacker_name": stacker_name})

# de-duplicate exact duplicates
spec_seen = set()
unique_specs = []
for spec in specs:
    key = json.dumps(
        {
            "family": spec["family"],
            "name": spec["name"],
            "models": spec["models"],
            "agg_method": spec.get("agg_method"),
            "trim_frac": spec.get("trim_frac"),
            "weight_scheme": spec.get("weight_scheme"),
            "weight_kwargs": spec.get("weight_kwargs"),
            "stacker_name": spec.get("stacker_name"),
        },
        sort_keys=True,
        ensure_ascii=False,
    )
    if key not in spec_seen:
        unique_specs.append(spec)
        spec_seen.add(key)

specs = unique_specs

# safety filter: keep only mixed specs
specs = [s for s in specs if modality_counts(s["models"])[2]]

print(f"Total cross-modal-only ensemble specs: {len(specs)}")

search_rows = []
for idx, spec in enumerate(specs, start=1):
    if idx % 500 == 0:
        print(f"Selection scoring {idx}/{len(specs)} ...")
    try:
        row = evaluate_spec_on_selection(spec, pred_fit_df, y_fit, pred_sel_df, y_sel)
        search_rows.append(row)
    except Exception as e:
        n_ml, n_dl, is_mixed = modality_counts(spec["models"])
        search_rows.append({
            "tag": spec_tag(spec),
            "family": spec["family"],
            "name": spec["name"],
            "n_models": len(spec["models"]),
            "n_ml_models": n_ml,
            "n_dl_models": n_dl,
            "is_mixed": is_mixed,
            "models": spec["models"],
            "selection_MAE": np.inf,
            "selection_RMSE": np.inf,
            "selection_MdAE": np.inf,
            "error": repr(e),
            "spec": spec,
        })

finite_search_rows = [r for r in search_rows if np.isfinite(r["selection_MAE"])]
if len(finite_search_rows) == 0:
    raise RuntimeError("No finite cross-modal ensemble specs were scored successfully.")

finite_search_rows = sorted(
    finite_search_rows,
    key=lambda r: (r["selection_MAE"], r["selection_RMSE"], r["selection_MdAE"], r["tag"])
)

search_df = pd.DataFrame([
    {
        "tag": r["tag"],
        "family": r["family"],
        "name": r["name"],
        "n_models": r["n_models"],
        "n_ml_models": r["n_ml_models"],
        "n_dl_models": r["n_dl_models"],
        "is_mixed": r["is_mixed"],
        "models": json.dumps(r["models"], ensure_ascii=False),
        "model_groups": json.dumps([model_group_map.get(m, "?") for m in r["models"]], ensure_ascii=False),
        "selection_MAE": r["selection_MAE"],
        "selection_RMSE": r["selection_RMSE"],
        "selection_MdAE": r["selection_MdAE"],
        "error": r.get("error", ""),
        "weights": json.dumps(r.get("weights", []), ensure_ascii=False),
        "weight_first": r.get("weight_first", np.nan),
    }
    for r in finite_search_rows
]).sort_values(["selection_MAE", "selection_RMSE", "selection_MdAE", "tag"], kind="stable").reset_index(drop=True)

# In this notebook every valid spec is mixed by construction.
mixed_search_df = search_df.copy()

save_df(search_df, "ensemble_search_leaderboard_cross_modal_only.csv")
save_df(mixed_search_df, "ensemble_search_leaderboard_mixed_only.csv")

display(search_df.head(30))

top_refit_overall = finite_search_rows[:min(REFIT_TOP_ENSEMBLES, len(finite_search_rows))]
top_refit_mixed = top_refit_overall[:min(REFIT_TOP_MIXED_ENSEMBLES, len(top_refit_overall))]

refit_union = {}
for r in top_refit_overall + top_refit_mixed:
    refit_union[r["tag"]] = r

top_refit_rows = list(refit_union.values())
top_refit_rows = sorted(
    top_refit_rows,
    key=lambda r: (r["selection_MAE"], r["selection_RMSE"], r["selection_MdAE"], r["tag"])
)

print(f"Top rows selected for holdout evaluation: {len(top_refit_rows)}")

In [ ]:

# ------------------------------------------------------------
# Holdout evaluation
# ------------------------------------------------------------
final_rows = []
for idx, row in enumerate(top_refit_rows, start=1):
    if idx % 50 == 0:
        print(f"Holdout eval {idx}/{len(top_refit_rows)} ...")
    spec = row["spec"]
    out = refit_and_eval_spec(
        spec,
        pred_val_df=val_pred_df,
        y_val=y_blend_val,
        pred_test_typ_df=test_typ_fullfit_pred_df,
        y_test_typ=y_test_typ,
        pred_test_full_df=test_full_fullfit_pred_df,
        y_test_full=y_test_full,
    )
    out["selection_MAE"] = row["selection_MAE"]
    out["selection_RMSE"] = row["selection_RMSE"]
    out["selection_MdAE"] = row["selection_MdAE"]
    final_rows.append(out)

if len(final_rows) == 0:
    raise RuntimeError("Нет строк для holdout-оценки.")

final_rows = sorted(
    final_rows,
    key=lambda r: (r["selection_MAE"], r["MAE_typical"], r["MAE_full"], r["tag"])
)

final_df = pd.DataFrame([
    {
        "tag": r["tag"],
        "family": r["family"],
        "name": r["name"],
        "n_models": r["n_models"],
        "n_ml_models": r["n_ml_models"],
        "n_dl_models": r["n_dl_models"],
        "is_mixed": r["is_mixed"],
        "models": json.dumps(r["models"], ensure_ascii=False),
        "model_groups": json.dumps([model_group_map.get(m, "?") for m in r["models"]], ensure_ascii=False),
        "selection_MAE": r["selection_MAE"],
        "selection_RMSE": r["selection_RMSE"],
        "selection_MdAE": r["selection_MdAE"],
        "MAE_typical": r["MAE_typical"],
        "RMSE_typical": r["RMSE_typical"],
        "MdAE_typical": r["MdAE_typical"],
        "MAE_full": r["MAE_full"],
        "RMSE_full": r["RMSE_full"],
        "MdAE_full": r["MdAE_full"],
        "weights": json.dumps(r.get("weights", []), ensure_ascii=False),
        "weights_like": json.dumps(r.get("weights_like", []), ensure_ascii=False),
        "weight_first": r.get("weight_first", np.nan),
    }
    for r in final_rows
]).sort_values(["selection_MAE", "MAE_typical", "MAE_full", "tag"], kind="stable").reset_index(drop=True)

mixed_final_df = final_df[final_df["is_mixed"] == True].reset_index(drop=True)

save_df(final_df, "ensemble_holdout_leaderboard.csv")
save_df(mixed_final_df, "ensemble_holdout_leaderboard_mixed_only.csv")

display(final_df.head(30))
display(mixed_final_df.head(30))

best_single_base = base_leaderboard.iloc[0]["base_id"]
best_single_ml = base_leaderboard[base_leaderboard["source_group"] == "ML"].iloc[0]["base_id"] if (base_leaderboard["source_group"] == "ML").any() else None
best_single_dl = base_leaderboard[base_leaderboard["source_group"] == "DL"].iloc[0]["base_id"] if (base_leaderboard["source_group"] == "DL").any() else None

single_typ = score_predictions(y_test_typ, test_typ_fullfit_pred_df[best_single_base].values)
single_full = score_predictions(y_test_full, test_full_fullfit_pred_df[best_single_base].values)

best_ensemble_row = final_rows[0]
best_mixed_row = next((r for r in final_rows if r["is_mixed"]), None)

summary_rows = [
    {"item": "best_single_base_by_val", "value": best_single_base},
    {"item": "best_single_ml_by_val", "value": best_single_ml},
    {"item": "best_single_dl_by_val", "value": best_single_dl},
    {"item": "best_ensemble_tag", "value": best_ensemble_row["tag"]},
    {"item": "best_ensemble_selection_MAE", "value": best_ensemble_row["selection_MAE"]},
    {"item": "best_ensemble_MAE_typical", "value": best_ensemble_row["MAE_typical"]},
    {"item": "best_ensemble_MAE_full", "value": best_ensemble_row["MAE_full"]},
]
if best_mixed_row is not None:
    summary_rows.extend([
        {"item": "best_mixed_ensemble_tag", "value": best_mixed_row["tag"]},
        {"item": "best_mixed_ensemble_selection_MAE", "value": best_mixed_row["selection_MAE"]},
        {"item": "best_mixed_ensemble_MAE_typical", "value": best_mixed_row["MAE_typical"]},
        {"item": "best_mixed_ensemble_MAE_full", "value": best_mixed_row["MAE_full"]},
    ])

summary_df = pd.DataFrame(summary_rows)
save_df(summary_df, "joint_summary_table.csv")
display(summary_df)

comparison_payload = {
    "selection_metric": "blend_select_MAE",
    "blend_train_frac_source": BLEND_TRAIN_FRAC_SOURCE,
    "blend_fit_frac": BLEND_FIT_FRAC,
    "ml_run_dir": str(ML_RUN_DIR),
    "dl_run_dir": str(DL_RUN_DIR),
    "n_base_learners_total": int(len(available_base_ids)),
    "n_base_learners_ml": int((base_pool_df["source_group"] == "ML").sum()),
    "n_base_learners_dl": int((base_pool_df["source_group"] == "DL").sum()),
    "available_base_learners": available_base_ids,
    "available_base_learners_ml": base_pool_df.loc[base_pool_df["source_group"] == "ML", "base_id"].tolist(),
    "available_base_learners_dl": base_pool_df.loc[base_pool_df["source_group"] == "DL", "base_id"].tolist(),
    "best_single_base_by_val": best_single_base,
    "best_single_ml_by_val": best_single_ml,
    "best_single_dl_by_val": best_single_dl,
    "best_single_base_metrics": {
        "MAE_typical": single_typ["MAE"],
        "RMSE_typical": single_typ["RMSE"],
        "MdAE_typical": single_typ["MdAE"],
        "MAE_full": single_full["MAE"],
        "RMSE_full": single_full["RMSE"],
        "MdAE_full": single_full["MdAE"],
    },
    "best_ensemble": {
        "tag": best_ensemble_row["tag"],
        "family": best_ensemble_row["family"],
        "name": best_ensemble_row["name"],
        "models": best_ensemble_row["models"],
        "n_ml_models": best_ensemble_row["n_ml_models"],
        "n_dl_models": best_ensemble_row["n_dl_models"],
        "is_mixed": best_ensemble_row["is_mixed"],
        "selection_MAE": best_ensemble_row["selection_MAE"],
        "MAE_typical": best_ensemble_row["MAE_typical"],
        "RMSE_typical": best_ensemble_row["RMSE_typical"],
        "MdAE_typical": best_ensemble_row["MdAE_typical"],
        "MAE_full": best_ensemble_row["MAE_full"],
        "RMSE_full": best_ensemble_row["RMSE_full"],
        "MdAE_full": best_ensemble_row["MdAE_full"],
        "weights": best_ensemble_row.get("weights", []),
        "weights_like": best_ensemble_row.get("weights_like", []),
        "weight_first": best_ensemble_row.get("weight_first", None),
    },
    "best_mixed_ensemble": None if best_mixed_row is None else {
        "tag": best_mixed_row["tag"],
        "family": best_mixed_row["family"],
        "name": best_mixed_row["name"],
        "models": best_mixed_row["models"],
        "n_ml_models": best_mixed_row["n_ml_models"],
        "n_dl_models": best_mixed_row["n_dl_models"],
        "selection_MAE": best_mixed_row["selection_MAE"],
        "MAE_typical": best_mixed_row["MAE_typical"],
        "RMSE_typical": best_mixed_row["RMSE_typical"],
        "MdAE_typical": best_mixed_row["MdAE_typical"],
        "MAE_full": best_mixed_row["MAE_full"],
        "RMSE_full": best_mixed_row["RMSE_full"],
        "MdAE_full": best_mixed_row["MdAE_full"],
        "weights": best_mixed_row.get("weights", []),
        "weights_like": best_mixed_row.get("weights_like", []),
        "weight_first": best_mixed_row.get("weight_first", None),
    },
}
save_json(comparison_payload, "best_ensemble_summary.json")

best_models = best_ensemble_row["models"]
best_weights = best_ensemble_row.get("weights") or best_ensemble_row.get("weights_like") or []
if len(best_weights) == len(best_models) and len(best_weights) > 0:
    wdf = pd.DataFrame({"base_id": best_models, "weight": best_weights})
    wdf = wdf.merge(
        base_pool_df[
            ["base_id", "source_group", "model", "provider", "scheme_family", "experiment", "base_id_original"]
        ],
        on="base_id",
        how="left",
    )
    save_df(wdf, "best_ensemble_weights.csv")

composition_summary_df = (
    final_df.assign(
        composition=np.where(
            final_df["n_ml_models"] == 0,
            "DL_only",
            np.where(final_df["n_dl_models"] == 0, "ML_only", "mixed"),
        )
    )
    .groupby("composition", dropna=False)
    .agg(
        n_ensembles=("tag", "count"),
        best_selection_MAE=("selection_MAE", "min"),
        best_holdout_typical_MAE=("MAE_typical", "min"),
        best_holdout_full_MAE=("MAE_full", "min"),
    )
    .reset_index()
)
save_df(composition_summary_df, "composition_summary.csv")
display(composition_summary_df)

run_manifest = {
    "run_name": RUN_NAME,
    "artifact_root": str(ARTIFACT_ROOT),
    "run_dir": str(RUN_DIR),
    "data_path": str(DATA_PATH),
    "target_col": target_col,
    "duration_cap": cap,
    "ml_run_dir": str(ML_RUN_DIR),
    "dl_run_dir": str(DL_RUN_DIR),
    "blend_train_frac_source": BLEND_TRAIN_FRAC_SOURCE,
    "blend_fit_frac": BLEND_FIT_FRAC,
    "n_selected_base_learners": int(len(available_base_ids)),
    "n_selected_base_learners_ml": int((base_pool_df["source_group"] == "ML").sum()),
    "n_selected_base_learners_dl": int((base_pool_df["source_group"] == "DL").sum()),
    "n_search_specs": int(len(specs)),
    "n_refit_rows": int(len(top_refit_rows)),
    "best_single_base_by_val": best_single_base,
    "best_ensemble_tag": best_ensemble_row["tag"],
    "best_mixed_ensemble_tag": None if best_mixed_row is None else best_mixed_row["tag"],
}
save_json(run_manifest, "run_manifest.json")

print("Сохранено:", RUN_DIR.resolve())


In [ ]:

# ------------------------------------------------------------
# Графики
# ------------------------------------------------------------
top_plot = final_df.head(20).copy()

def composition_color(row):
    if row["n_ml_models"] == 0:
        return "tab:blue"     # DL-only (should not appear here)
    if row["n_dl_models"] == 0:
        return "tab:orange"   # ML-only (should not appear here)
    return "tab:green"        # mixed

plt.figure(figsize=(12, 8))
colors = top_plot.apply(composition_color, axis=1)
plt.barh(top_plot["tag"][::-1], top_plot["MAE_typical"][::-1], color=list(colors[::-1]))
plt.xlabel("MAE_typical")
plt.ylabel("Ensemble")
plt.title("Top-20 joint ML+DL ensembles by selection_MAE -> holdout typical")
plt.tight_layout()
plt.show()

top_corr_models = ranked_models[:min(12, len(ranked_models))]
corr = val_pred_df[top_corr_models].corr()
save_df(corr.reset_index().rename(columns={"index": "base_id"}), "blend_val_prediction_correlation.csv")

plt.figure(figsize=(10, 8))
plt.imshow(corr.values, aspect="auto")
plt.colorbar()
plt.xticks(range(len(top_corr_models)), top_corr_models, rotation=90)
plt.yticks(range(len(top_corr_models)), top_corr_models)
plt.title("Correlation of blend-val predictions (top-12 base learners)")
plt.tight_layout()
plt.show()

print("Top overall:")
display(final_df.head(20))

print("Top mixed-only:")
display(mixed_final_df.head(20))

print("Top base learners by source:")
display(base_leaderboard.groupby("source_group", dropna=False).head(10).reset_index(drop=True))


## Что смотреть после запуска

Главные артефакты этого joint cross-modal-only эксперимента:

- `source_run_registry.csv` — какие именно ML и DL run dirs были найдены;
- `combined_available_base_meta.csv` — объединённый пул base learners;
- `joint_base_pred_blend_val.csv` — joint prediction matrix на blend-val;
- `ensemble_search_leaderboard_cross_modal_only.csv` — leaderboard **только для ансамблей ML+DL**;
- `ensemble_search_leaderboard_mixed_only.csv` — дублирующий mixed-only leaderboard;
- `final_joint_ensemble_leaderboard.csv` — финальная holdout-оценка лучших cross-modal ансамблей;
- `final_joint_ensemble_summary.json` — краткая сводка по лучшему ансамблю;
- `best_joint_ensemble_weights.csv` — веса лучшего ансамбля, если они интерпретируемы;
- `joint_ensemble_top20.png` — график топ-20 cross-modal ансамблей.

В этом ноутбуке валидные ансамбли по определению содержат и ML, и DL модели одновременно.